# 01 Data assembly: grid, fault labels, and geophysical features

This notebook builds the analysis grid, the fault labels, and the geophysical feature stack for the GeoDAWN region in western Nevada. Notebooks 02 and 03 read its outputs.

**Inputs.** All inputs are public.

* USGS GeoDAWN airborne magnetic and radiometric grids (ScienceBase).
* USGS Quaternary Fault and Fold Database, file `Qfaults_GIS.zip` (ScienceBase).
* USGS ComCat earthquake catalog, through the FDSN event web service.
* Nevada Geodetic Laboratory (NGL) MIDAS GNSS velocities.
* Optional: other rasters that you supply, for example isostatic gravity or INGENIOUS layers. Make sure that the license lets you use them.

**Outputs.** The notebook writes these objects under `STORE_ROOT`:

| Object | Content |
|---|---|
| `grid.json` | Grid definition (EPSG:32611, 100 m) |
| `features.zarr` | Feature stack and the `valid` mask |
| `labels.zarr` | Fault raster and distance to the nearest mapped fault |
| `qfaults_aoi.gpkg` | Quaternary fault traces in the area of interest |

**Annotations.**

* `VERIFY:` marks an assumption about an external service or file format. I did not test these against the live services. Check them on the first run.
* `NOTE:` marks a design decision that you can change.

**Behavior on errors.** The notebook stops with a clear error when an input is missing or malformed. It does not fall back silently.

**Environment.** Pangeo-based GeoLab image with `rioxarray`, `geopandas`, `pyogrio`, `scipy`, `requests`, `zarr`, and `fsspec`. Add `s3fs` if `STORE_ROOT` is an S3 URL.

In [ ]:
import io
import json
import os
import pathlib
import re
import warnings
import zipfile

import fsspec
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyogrio
import rasterio
import requests
import rioxarray  # noqa: F401  (registers the .rio accessor)
import xarray as xr
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.transform import from_origin
from rasterio.warp import transform_bounds
from pyproj import Transformer
from scipy import ndimage
from scipy.interpolate import RBFInterpolator
from scipy.spatial import cKDTree
from shapely.geometry import box

## 0. Configuration

Change the values in the next cell. The other cells use these values only.

In [ ]:
# Storage. NOTE: a local path works for all three notebooks, because only the notebook
# server writes to the store. Use an S3 URL (s3://bucket/prefix) to share results.
STORE_ROOT = os.environ.get("GEMS_STORE", str(pathlib.Path.home() / "gems"))
CACHE_DIR = pathlib.Path(os.environ.get("GEMS_CACHE", "/tmp/gems_cache"))

# Analysis grid. NOTE: same CRS and cell size as the GEMS challenge and the INGENIOUS products.
CRS = "EPSG:32611"
RES = 100.0

# ScienceBase items.
GEODAWN_ITEM = "657e1d85d34e23d3533209f7"   # GeoDAWN magnetic and radiometric surveys
QFAULTS_ITEM = "589097b1e4b072a7ac0cae23"   # Quaternary Fault and Fold Database for the Nation
QFAULTS_FILE = "Qfaults_GIS.zip"
QFAULTS_LAYER_FILTER = None                 # regex on layer name; None = keep every line layer

# Optional AOI override (lon_min, lat_min, lon_max, lat_max). None = GeoDAWN bounding box.
AOI_LONLAT_OVERRIDE = None

# GeoDAWN file selection (regex on file name). None = no filter.
GEODAWN_INCLUDE = None
GEODAWN_EXCLUDE = r"(?i)(flight|line|xyz|csv|pdf|xml|txt|metadata)"
MIN_VALID_FRACTION = 0.01                    # drop layers that cover less than 1 % of the grid

# Layers that get potential-field derivatives (regex on layer name).
POTENTIAL_FIELD_PATTERN = r"(?i)(mag|tmi|rtp|grav|bouguer|iso)"

# Earthquakes: USGS ComCat through the FDSN event service.
FDSN_EVENT_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
FDSN_LIMIT = 20000                           # VERIFY: current per-request limit of the service
EQ_START, EQ_END = "1980-01-01", "2026-09-01"
EQ_MIN_MAG = 1.0
EQ_SHALLOW_KM = 10.0
EQ_SMOOTH_M = 2000.0

# GNSS velocities (NGL MIDAS). Column layout is documented in midas.readme.txt at NGL.
# VERIFY: file name inside https://geodesy.unr.edu/gps_timeseries/IGS20/midas/
MIDAS_URL = "https://geodesy.unr.edu/gps_timeseries/IGS20/midas/midas.IGS20.txt"
GNSS_MIN_YEARS = 2.5
GNSS_MAX_SIGMA_MM = 1.0
GNSS_STRAIN_STEP_M = 2000.0
GNSS_RBF_SMOOTHING = 1.0      # NOTE: controls strain roughness. Compare with a published strain model.
GNSS_MAX_GAP_M = 50000.0      # strain is masked farther than this from the nearest station
AOI_BUFFER_M = 25000.0        # margin for GNSS and earthquakes around the grid

# Optional extra rasters: {"layer_name": "path or URL that rioxarray can open"}.
EXTRA_RASTERS = {
    # "iso_gravity": "s3://my-bucket/gravity/nv_isostatic_gravity.tif",
}
# Optional extra fault vectors (for example INGENIOUS): {"source_name": "path"}.
EXTRA_FAULT_VECTORS = {
    # "ingenious": "/home/jovyan/data/ingenious_faults.gpkg",
}

In [ ]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)
fs, _ = fsspec.core.url_to_fs(STORE_ROOT)
fs.makedirs(STORE_ROOT, exist_ok=True)

def store_path(rel):
    return f"{STORE_ROOT.rstrip('/')}/{rel}"

def put_file(local, rel):
    fs.put(str(local), store_path(rel))

print("STORE_ROOT:", STORE_ROOT)
print("CACHE_DIR: ", CACHE_DIR)

## 1. Helper functions for ScienceBase and downloads

These functions use the ScienceBase JSON API directly. They do not need `sciencebasepy`.

In [ ]:
SB_API = "https://www.sciencebase.gov/catalog"

def sb_item(item_id):
    r = requests.get(f"{SB_API}/item/{item_id}", params={"format": "json"}, timeout=60)
    r.raise_for_status()
    return r.json()

def sb_child_ids(item_id, page=1000):
    # VERIFY: ScienceBase paging. The function stops if one page is full.
    r = requests.get(f"{SB_API}/items",
                     params={"parentId": item_id, "format": "json", "max": page, "fields": "id"},
                     timeout=60)
    r.raise_for_status()
    items = r.json().get("items", [])
    if len(items) >= page:
        raise RuntimeError(f"Item {item_id} has {page} or more children. Add paging to sb_child_ids.")
    return [it["id"] for it in items]

def sb_files(item_id, recursive=True):
    it = sb_item(item_id)
    files = list(it.get("files", []))
    for facet in it.get("facets", []):          # VERIFY: some items keep files under "facets"
        files += facet.get("files", [])
    out = []
    for f in files:
        url = f.get("url") or f.get("downloadUri")
        if url:
            out.append({"item": item_id, "item_title": it.get("title", ""),
                        "name": f.get("name", ""), "url": url, "size": f.get("size")})
    if recursive:
        for cid in sb_child_ids(item_id):
            out += sb_files(cid, recursive=True)
    return out

def download(url, dest):
    dest = pathlib.Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_name(dest.name + ".part")
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        # ScienceBase can answer 200 with an HTML page (login, landing or error page) instead of the file.
        ctype = r.headers.get("Content-Type", "")
        if "text/html" in ctype:
            raise RuntimeError(f"{url} returned an HTML page ({ctype}), not a file. Open it in a browser to see why.")
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    tmp.rename(dest)
    return dest

def safe_name(s):
    return re.sub(r"[^0-9a-zA-Z]+", "_", s).strip("_").lower()

## 2. Analysis grid

The grid covers the GeoDAWN bounding box, projected to UTM zone 11N and snapped to 100 m. The `valid` mask (section 7) marks the cells inside the GeoDAWN survey coverage. Notebook 03 trains and predicts only inside that mask.

In [ ]:
if AOI_LONLAT_OVERRIDE is not None:
    AOI_LONLAT = tuple(AOI_LONLAT_OVERRIDE)
else:
    bb = sb_item(GEODAWN_ITEM).get("spatial", {}).get("boundingBox")
    if not bb:
        raise RuntimeError("The GeoDAWN item has no bounding box. Set AOI_LONLAT_OVERRIDE.")
    AOI_LONLAT = (bb["minX"], bb["minY"], bb["maxX"], bb["maxY"])
print("AOI (lon/lat):", AOI_LONLAT)

xmin, ymin, xmax, ymax = transform_bounds("EPSG:4326", CRS, *AOI_LONLAT, densify_pts=51)
xmin, ymin = float(np.floor(xmin / RES) * RES), float(np.floor(ymin / RES) * RES)
xmax, ymax = float(np.ceil(xmax / RES) * RES), float(np.ceil(ymax / RES) * RES)
W, H = int(round((xmax - xmin) / RES)), int(round((ymax - ymin) / RES))
TRANSFORM = from_origin(xmin, ymax, RES, RES)
GRID = {"crs": CRS, "res": RES, "xmin": xmin, "ymax": ymax, "width": W, "height": H,
        "aoi_lonlat": [float(v) for v in AOI_LONLAT]}

def make_template(grid):
    x = grid["xmin"] + grid["res"] * (np.arange(grid["width"]) + 0.5)
    y = grid["ymax"] - grid["res"] * (np.arange(grid["height"]) + 0.5)
    da = xr.DataArray(np.full((grid["height"], grid["width"]), np.nan, np.float32),
                      coords={"y": y, "x": x}, dims=("y", "x"))
    tr = from_origin(grid["xmin"], grid["ymax"], grid["res"], grid["res"])
    return da.rio.write_crs(grid["crs"]).rio.write_transform(tr)

template = make_template(GRID)
with fs.open(store_path("grid.json"), "w") as f:
    json.dump(GRID, f, indent=2)

# Buffered AOI in lon/lat for the point data queries.
AOI_BUF_LONLAT = transform_bounds(CRS, "EPSG:4326", xmin - AOI_BUFFER_M, ymin - AOI_BUFFER_M,
                                  xmax + AOI_BUFFER_M, ymax + AOI_BUFFER_M, densify_pts=51)
to_utm = Transformer.from_crs("EPSG:4326", CRS, always_xy=True)
print(f"Grid: {W} x {H} cells ({W * H / 1e6:.1f} M), {W * H * RES * RES / 1e6:,.0f} km2")

## 3. Fault labels

The Quaternary Fault and Fold Database is the label source. It is incomplete and some traces are offset from the true surface trace. Notebook 03 treats unmapped cells as unlabeled, not as confirmed negatives. For that reason, this section also saves the distance from every cell to the nearest mapped fault.

In [ ]:
qf = [f for f in sb_files(QFAULTS_ITEM, recursive=False) if f["name"] == QFAULTS_FILE]
if not qf:
    raise RuntimeError(f"{QFAULTS_FILE} is not attached to ScienceBase item {QFAULTS_ITEM}.")
qf_zip = download(qf[0]["url"], CACHE_DIR / QFAULTS_FILE)
qf_dir = CACHE_DIR / "qfaults"
if not qf_dir.exists():
    with zipfile.ZipFile(qf_zip) as z:
        z.extractall(qf_dir)

sources = [p for p in qf_dir.rglob("*") if p.suffix.lower() in (".shp", ".gpkg", ".gdb")]
line_layers = []
for src in sources:
    for name, gtype in pyogrio.list_layers(src):
        if gtype and "line" in str(gtype).lower():
            if QFAULTS_LAYER_FILTER is None or re.search(QFAULTS_LAYER_FILTER, name):
                line_layers.append((src, name))
if not line_layers:
    raise RuntimeError(f"No line layers found in {qf_dir}. Inspect the archive.")
print("Line layers used:")
for src, name in line_layers:
    print(f"  {src.name} :: {name}")
if len(line_layers) > 1:
    # VERIFY: if the archive holds detailed and simplified traces, keep only the detailed layer.
    warnings.warn("More than one line layer is used. Set QFAULTS_LAYER_FILTER to avoid duplicates.")

In [ ]:
aoi_box = box(xmin, ymin, xmax, ymax)

def load_lines(path, layer=None, source=""):
    g = gpd.read_file(path, layer=layer)
    if g.crs is None:
        raise RuntimeError(f"{path} has no CRS.")
    g = g.to_crs(CRS)
    g = g[g.intersects(aoi_box)].copy()
    g["geometry"] = g.geometry.intersection(aoi_box)
    g = g[~g.geometry.is_empty]
    # GeoPackage field names are case-insensitive: "FAULT_NAME" and "fault_name" from
    # different layers would collide on write. Lowercase them and keep the first of any duplicates.
    g = g.rename_geometry("geometry") if g.geometry.name != "geometry" else g
    g.columns = [c.lower() for c in g.columns]
    g = g.loc[:, ~g.columns.duplicated()]
    g["label_source"] = source
    return g

qfaults = pd.concat([load_lines(s, n, "qfaults") for s, n in line_layers], ignore_index=True)
qfaults = gpd.GeoDataFrame(qfaults, geometry="geometry", crs=CRS)
print(len(qfaults), "Quaternary fault features in the AOI")

extra = [load_lines(p, None, k) for k, p in EXTRA_FAULT_VECTORS.items()]
all_faults = gpd.GeoDataFrame(pd.concat([qfaults] + extra, ignore_index=True), geometry="geometry", crs=CRS)

local = CACHE_DIR / "qfaults_aoi.gpkg"
all_faults.to_file(local, driver="GPKG")
put_file(local, "qfaults_aoi.gpkg")

def burn(gdf):
    if len(gdf) == 0:
        return np.zeros((H, W), np.uint8)
    # NOTE: all_touched=True marks every cell that a trace crosses.
    return rasterize(((g, 1) for g in gdf.geometry), out_shape=(H, W), transform=TRANSFORM,
                     fill=0, all_touched=True, dtype="uint8")

fault_q = burn(qfaults)
fault_all = burn(all_faults)
fault_dist_m = (ndimage.distance_transform_edt(fault_all == 0) * RES).astype(np.float32)
print(f"Fault cells: {fault_all.sum():,} ({fault_all.mean() * 100:.2f} % of the grid)")

## 4. GeoDAWN grids

The next cell lists all files on the GeoDAWN item and its child items. Look at the table, then set `GEODAWN_INCLUDE` and `GEODAWN_EXCLUDE` in section 0 if necessary.

`VERIFY:` the GeoDAWN grid format. The code opens every file that GDAL can read as a single-band raster. If the grids use a format that GDAL cannot read (for example some Geosoft formats), the cell stops. Convert those grids to GeoTIFF first.

In [ ]:
gd_files = pd.DataFrame(sb_files(GEODAWN_ITEM, recursive=True))
if gd_files.empty:
    raise RuntimeError("No files found on the GeoDAWN item.")
print(len(gd_files), "files on the GeoDAWN item tree")
display(gd_files[["item_title", "name", "size"]])

RASTER_EXT = (".tif", ".tiff", ".zip", ".grd", ".ers", ".img", ".gxf")
sel = gd_files[gd_files["name"].str.lower().str.endswith(RASTER_EXT)]
if GEODAWN_EXCLUDE:
    sel = sel[~sel["name"].str.contains(GEODAWN_EXCLUDE, regex=True)]
if GEODAWN_INCLUDE:
    sel = sel[sel["name"].str.contains(GEODAWN_INCLUDE, regex=True)]
print(f"Selected {len(sel)} files, {pd.to_numeric(sel['size']).fillna(0).sum() / 1e9:.2f} GB")

In [ ]:
SKIP_EXT = {".xml", ".txt", ".pdf", ".csv", ".prj", ".aux", ".ovr", ".xyz", ".dbf", ".shx", ".shp", ".html", ".htm"}

def readable_rasters(path):
    path = pathlib.Path(path)
    if path.suffix.lower() == ".zip":
        out_dir = path.with_suffix("")
        if not zipfile.is_zipfile(path):
            size, head = path.stat().st_size, path.read_bytes()[:200]
            path.unlink()   # remove the bad cached copy so the next run downloads it again
            raise RuntimeError(f"{path} is not a zip file ({size:,} bytes, cached copy deleted). "
                               f"First bytes: {head!r}")
        if not out_dir.exists():
            with zipfile.ZipFile(path) as z:
                z.extractall(out_dir)
        paths = [p for p in out_dir.rglob("*") if p.is_file()]
    else:
        paths = [path]
    found = []
    for p in paths:
        if p.suffix.lower() in SKIP_EXT:
            continue
        try:
            with rasterio.open(p) as src:
                if src.count == 1 and src.width > 10 and src.height > 10:
                    found.append(p)
        except rasterio.errors.RasterioIOError:
            pass
    return found

def to_grid(src, name):
    da = rioxarray.open_rasterio(src, masked=True)
    if "band" in da.dims:
        da = da.squeeze("band", drop=True)
    if da.rio.crs is None:
        raise RuntimeError(f"{name}: {src} has no CRS. Assign one with .rio.write_crs() first.")
    res = abs(da.rio.resolution()[0])
    res_m = res * 111_000.0 if da.rio.crs.is_geographic else res   # VERIFY: projected units are metres
    # NOTE: average when the source is finer than the grid, bilinear when it is coarser.
    method = Resampling.average if res_m < RES else Resampling.bilinear
    out = da.astype("float32").rio.reproject_match(template, resampling=method)
    out = out.assign_coords(x=template.x, y=template.y)
    return out.rename(name)

layers = {}
geodawn_names = []
for _, row in sel.iterrows():
    local = download(row["url"], CACHE_DIR / "geodawn" / row["item"] / row["name"])
    for p in readable_rasters(local):
        name = "gd_" + safe_name(p.stem)
        while name in layers:
            name += "_b"
        da = to_grid(p, name)
        frac = float(np.isfinite(da.values).mean())
        if frac < MIN_VALID_FRACTION:
            print(f"  skip {name}: {frac:.3%} valid")
            continue
        layers[name] = da
        geodawn_names.append(name)
        print(f"  {name:45s} {frac:6.1%} valid   from {p.name}")
if not geodawn_names:
    raise RuntimeError("No readable GeoDAWN rasters. Check the file table and the selection patterns.")

## 5. Extra rasters and potential-field derivatives

The derivatives are the horizontal gradient magnitude (`thg`), the first vertical derivative (`vdr`, FFT method), and the tilt angle (`tilt`). Maxima of `thg` and zero crossings of `tilt` often mark buried contacts and faults that have no surface scarp.

In [ ]:
for name, src in EXTRA_RASTERS.items():
    layers[name] = to_grid(src, name)
    print(f"  {name}: {float(np.isfinite(layers[name].values).mean()):.1%} valid")

def potential_field_derivatives(z, dx, pad=64):
    """Horizontal gradient magnitude, first vertical derivative, and tilt angle.

    NOTE: NaN cells are filled with the nearest valid value before the FFT.
    Results are set back to NaN at those cells. Expect edge effects near large gaps.
    """
    valid = np.isfinite(z)
    if valid.sum() < 100:
        raise ValueError("Too few valid cells for derivatives.")
    idx = ndimage.distance_transform_edt(~valid, return_distances=False, return_indices=True)
    zf = z[tuple(idx)].astype(np.float64)
    zf -= zf.mean()
    zp = np.pad(zf, pad, mode="reflect")
    ny, nx = zp.shape
    kx = 2 * np.pi * np.fft.fftfreq(nx, d=dx)
    ky = 2 * np.pi * np.fft.fftfreq(ny, d=dx)
    kk = np.hypot(*np.meshgrid(kx, ky))
    vdr = np.real(np.fft.ifft2(np.fft.fft2(zp) * kk))[pad:-pad, pad:-pad]
    gr, gc = np.gradient(zf, dx)
    thg = np.hypot(gr, gc)
    tilt = np.arctan2(vdr, thg)
    return {k: np.where(valid, v, np.nan).astype(np.float32)
            for k, v in {"thg": thg, "vdr": vdr, "tilt": tilt}.items()}

for name in [n for n in list(layers) if re.search(POTENTIAL_FIELD_PATTERN, n)]:
    for suffix, arr in potential_field_derivatives(layers[name].values, RES).items():
        layers[f"{name}_{suffix}"] = template.copy(data=arr).rename(f"{name}_{suffix}")
    print("  derivatives for", name)

## 6. Earthquake density and GNSS strain rate

**Earthquakes.** The query splits a time window in two when the service returns its maximum number of events. So large aftershock sequences do not truncate the catalog without notice. The result is two log-density layers: all events, and events shallower than `EQ_SHALLOW_KM`.

**GNSS.** The MIDAS east and north velocities are interpolated with a smoothed thin-plate spline. The strain rate tensor comes from the spatial gradients of that field. The reference frame does not change the strain rate, because a rigid rotation adds rotation only. `NOTE:` this is a simple estimator. Compare its output with a published strain rate model for the Great Basin before you rely on it.

In [ ]:
def fetch_comcat(bbox_lonlat, t0, t1, min_mag):
    lon0, lat0, lon1, lat1 = bbox_lonlat
    t0, t1 = pd.Timestamp(t0), pd.Timestamp(t1)
    params = {"format": "csv", "starttime": t0.isoformat(), "endtime": t1.isoformat(),
              "minlatitude": lat0, "maxlatitude": lat1, "minlongitude": lon0, "maxlongitude": lon1,
              "minmagnitude": min_mag, "orderby": "time-asc", "limit": FDSN_LIMIT}
    r = requests.get(FDSN_EVENT_URL, params=params, timeout=300)
    if r.status_code == 204 or not r.text.strip():
        return pd.DataFrame()
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text))
    if len(df) >= FDSN_LIMIT:
        mid = t0 + (t1 - t0) / 2
        return pd.concat([fetch_comcat(bbox_lonlat, t0, mid, min_mag),
                          fetch_comcat(bbox_lonlat, mid, t1, min_mag)], ignore_index=True)
    return df

eq_parts = []
for y0 in range(pd.Timestamp(EQ_START).year, pd.Timestamp(EQ_END).year + 1):
    t0 = max(pd.Timestamp(f"{y0}-01-01"), pd.Timestamp(EQ_START))
    t1 = min(pd.Timestamp(f"{y0 + 1}-01-01"), pd.Timestamp(EQ_END))
    if t1 > t0:
        eq_parts.append(fetch_comcat(AOI_BUF_LONLAT, t0, t1, EQ_MIN_MAG))
eq = pd.concat(eq_parts, ignore_index=True).drop_duplicates("id")
if eq.empty:
    raise RuntimeError("No earthquakes returned. Check the AOI and the service.")
eq["x"], eq["y"] = to_utm.transform(eq["longitude"].values, eq["latitude"].values)
print(f"{len(eq):,} events, M >= {EQ_MIN_MAG}, {EQ_START} to {EQ_END}")

In [ ]:
def point_log_density(x, y, sigma_m, grid):
    """log10 of kernel-smoothed point density (events per km2) on the grid."""
    res, h, w = grid["res"], grid["height"], grid["width"]
    pad = int(np.ceil(3 * sigma_m / res))
    col = np.floor((np.asarray(x) - grid["xmin"]) / res).astype(int) + pad
    row = np.floor((grid["ymax"] - np.asarray(y)) / res).astype(int) + pad
    ok = (col >= 0) & (col < w + 2 * pad) & (row >= 0) & (row < h + 2 * pad)
    counts = np.zeros((h + 2 * pad, w + 2 * pad), np.float64)
    np.add.at(counts, (row[ok], col[ok]), 1.0)
    dens = ndimage.gaussian_filter(counts, sigma=sigma_m / res, mode="constant")
    dens = dens[pad:pad + h, pad:pad + w] / (res * res / 1e6)
    return np.log10(dens + 1e-3).astype(np.float32)

In [ ]:
layers["eq_logdens_all"] = template.copy(data=point_log_density(eq.x, eq.y, EQ_SMOOTH_M, GRID)).rename("eq_logdens_all")
shallow = eq[eq["depth"] <= EQ_SHALLOW_KM]
layers["eq_logdens_shallow"] = template.copy(data=point_log_density(shallow.x, shallow.y, EQ_SMOOTH_M, GRID)).rename("eq_logdens_shallow")
print(f"{len(shallow):,} events shallower than {EQ_SHALLOW_KM} km")

In [ ]:
MIDAS_COLS = ["sta", "version", "t0", "t1", "dur", "n_epochs", "n_good", "n_pairs",
              "ve", "vn", "vu", "se", "sn", "su", "oe", "on", "ou", "fe", "fn", "fu",
              "sde", "sdn", "sdu", "n_steps", "lat", "lon", "h"]

r = requests.get(MIDAS_URL, timeout=300)
r.raise_for_status()
mid = pd.read_csv(io.StringIO(r.text), sep=r"\s+", header=None, comment="#", engine="python")
if mid.shape[1] < 27:
    raise RuntimeError(f"MIDAS file has {mid.shape[1]} columns, expected 27. Check MIDAS_URL.")
mid = mid.iloc[:, :27]
mid.columns = MIDAS_COLS
mid["lon"] = ((mid["lon"] + 180.0) % 360.0) - 180.0   # NOTE: NGL files can list longitudes outside -180..180
if not mid["lat"].between(-90, 90).all():
    raise RuntimeError("Column 25 is not latitude. The MIDAS format changed.")

lon0, lat0, lon1, lat1 = AOI_BUF_LONLAT
gn = mid[mid.lon.between(lon0, lon1) & mid.lat.between(lat0, lat1)
         & (mid.dur >= GNSS_MIN_YEARS)
         & (mid.se * 1e3 <= GNSS_MAX_SIGMA_MM) & (mid.sn * 1e3 <= GNSS_MAX_SIGMA_MM)].copy()
gn["x"], gn["y"] = to_utm.transform(gn.lon.values, gn.lat.values)
print(f"{len(gn)} GNSS stations after filtering")
if len(gn) < 10:
    raise RuntimeError("Fewer than 10 GNSS stations. Check the AOI, MIDAS_URL, and filters.")

In [ ]:
def strain_rate_field(xs, ys, ve_mm, vn_mm, gx, gy, smoothing):
    """Strain rate (nanostrain/yr) from station velocities (mm/yr).

    Coordinates go to km, so gradients are (mm/yr)/km = 1e-6 /yr = 1000 nanostrain/yr.
    gy runs from north to south (image order). np.gradient uses the real coordinates,
    so the signs are correct.
    """
    pts = np.column_stack([xs, ys]) / 1e3
    XX, YY = np.meshgrid(gx, gy)
    q = np.column_stack([XX.ravel(), YY.ravel()]) / 1e3
    fe = RBFInterpolator(pts, ve_mm, kernel="thin_plate_spline", smoothing=smoothing)(q).reshape(XX.shape)
    fn = RBFInterpolator(pts, vn_mm, kernel="thin_plate_spline", smoothing=smoothing)(q).reshape(XX.shape)
    dve_dy, dve_dx = np.gradient(fe, gy / 1e3, gx / 1e3)
    dvn_dy, dvn_dx = np.gradient(fn, gy / 1e3, gx / 1e3)
    s = 1e3
    exx, eyy, exy = dve_dx * s, dvn_dy * s, 0.5 * (dve_dy + dvn_dx) * s
    return {"dilatation": exx + eyy,
            "max_shear": np.sqrt(((exx - eyy) / 2) ** 2 + exy ** 2),
            "second_inv": np.sqrt(exx ** 2 + eyy ** 2 + 2 * exy ** 2)}

In [ ]:
step = GNSS_STRAIN_STEP_M
gx = np.arange(xmin, xmax + step, step)
gy = np.arange(ymax, ymin - step, -step)
strain = strain_rate_field(gn.x.values, gn.y.values, gn.ve.values * 1e3, gn.vn.values * 1e3,
                           gx, gy, GNSS_RBF_SMOOTHING)
gap, _ = cKDTree(np.column_stack([gn.x, gn.y])).query(
    np.column_stack([a.ravel() for a in np.meshgrid(gx, gy)]))
gap = gap.reshape(len(gy), len(gx))
for k, v in strain.items():
    coarse = xr.DataArray(np.where(gap <= GNSS_MAX_GAP_M, v, np.nan), coords={"y": gy, "x": gx}, dims=("y", "x"))
    fine = coarse.interp(x=template.x, y=template.y, method="linear").astype("float32")
    layers[f"gnss_{k}"] = template.copy(data=fine.values).rename(f"gnss_{k}")
    print(f"  gnss_{k}: median {np.nanmedian(fine.values):.1f} nanostrain/yr")

## 7. Write the feature stack and the labels

In [ ]:
ds = xr.Dataset({k: v.astype("float32") for k, v in layers.items()})
valid = np.zeros((H, W), bool)
for n in geodawn_names:
    valid |= np.isfinite(ds[n].values)
ds["valid"] = (("y", "x"), valid.astype("uint8"))
ds = ds.rio.write_crs(CRS)
ds.attrs.update({"grid": json.dumps(GRID), "geodawn_layers": json.dumps(geodawn_names)})
ds.chunk({"y": 1024, "x": 1024}).to_zarr(store_path("features.zarr"), mode="w", consolidated=True)

lab = xr.Dataset({"fault": (("y", "x"), fault_all), "fault_qfaults": (("y", "x"), fault_q),
                  "fault_dist_m": (("y", "x"), fault_dist_m)},
                 coords={"y": template.y, "x": template.x}).rio.write_crs(CRS)
lab.attrs["grid"] = json.dumps(GRID)
lab.chunk({"y": 1024, "x": 1024}).to_zarr(store_path("labels.zarr"), mode="w", consolidated=True)

print(f"features.zarr: {len(layers)} layers, valid area {valid.mean():.1%} of the grid")
print(f"labels.zarr:   {int((fault_all & valid).sum()):,} fault cells inside the valid mask")

In [ ]:
show = geodawn_names[:2] + ["eq_logdens_shallow", "gnss_dilatation"]
fig, axes = plt.subplots(1, len(show), figsize=(5 * len(show), 5), constrained_layout=True)
for ax, n in zip(np.atleast_1d(axes), show):
    v = ds[n].values
    lo, hi = np.nanpercentile(v, [2, 98])
    ax.imshow(v, vmin=lo, vmax=hi, cmap="viridis")
    ax.contour(fault_all, levels=[0.5], colors="w", linewidths=0.3)
    ax.set_title(n)
    ax.set_axis_off()